# Configure Codex for regulated enterprise environments

Financial services, healthcare, public-sector, and other regulated organizations need more than a filesystem policy when they introduce an AI coding assistant. They need an operating model that covers identity, human oversight, permitted execution modes, network access, integrations, sensitive data, monitoring, change management, and device deployment.

This cookbook provides a general-purpose starting point for the latest supported local Codex desktop, CLI, and IDE clients. It includes one complete enterprise `requirements.toml`, one matching client `config.toml`, managed model and identity governance, enterprise configuration guidance, and a practical rollout checklist.

The examples are technical deployment guidance, not a compliance certification, legal interpretation, contractual data-processing commitment, or substitute for an organization-specific risk assessment.

## What you will learn

By the end of this cookbook, you will be able to:

- Explain the difference between enforced enterprise requirements and changeable client defaults.
- Start with one portable `requirements.toml` and one matching `config.toml`.
- Set an approved baseline for sign-in, human review, execution boundaries, networking, integrations, secrets, and observability.
- Restrict visible models and reasoning options for approved enterprise groups.
- Apply native Windows sandbox settings and macOS managed configuration without maintaining separate platform-specific files.
- Choose an approved enterprise-managed configuration approach.
- Validate the configuration, pilot the rollout, and collect actionable support feedback.

## Before you begin

To follow this cookbook, you need:

- An approved ChatGPT Enterprise or regulated-workspace deployment and the authority to configure its managed Codex policies.
- The latest approved local Codex desktop, CLI, or IDE release across all managed devices.
- The appropriate enterprise identity, device-management, security, and change-management approvals.

Review the example files locally. Model-catalog export is a separate action that uses an already authorized Codex account.

## Download the starter files

Use these two reviewed reference files as the starting point for an enterprise deployment:

- [Download the enterprise requirements.toml](regulated_industry_configuration/requirements.toml) - The managed policy that defines supported, non-overridable boundaries.
- [Download the matching config.toml](regulated_industry_configuration/config.toml) - The recommended client defaults used inside those boundaries.

Keep the two TOML files paired. Do not assume that a setting becomes enforced simply because it appears in client configuration or managed defaults.

## Follow the deployment process

Use the sections below as one end-to-end enterprise rollout:

1. Define the organization's risk groups, identity boundaries, and required human oversight.
2. Review the linked `requirements.toml` and remove any settings the organization cannot support.
3. Pair it with `config.toml` and confirm that every default remains inside the managed policy.
4. Configure approved model catalogs and enterprise login restrictions on fully updated clients.
5. Confirm the Windows sandbox requirements and macOS management approach for the supported devices.
6. Deploy the same two shared files through an approved enterprise-managed workflow.
7. Validate the effective configuration, run a limited pilot, and confirm enterprise support and monitoring.

The rest of the cookbook explains each choice and provides the complete reference files and deployment guidance.

## Separate enterprise requirements from client defaults

Codex configuration operates across two layers:

| Layer | File | Purpose | Can the user override it? |
| --- | --- | --- | --- |
| Enterprise requirements | `requirements.toml` | Establish supported, non-overridable policy boundaries. | No, when delivered through a supported managed-requirements source. |
| Client configuration | `config.toml` | Establish client behavior, preferred defaults, and platform settings. | Yes, unless a matching requirement or external administrative control constrains the setting. |

Managed defaults delivered through `managed_config.toml` or macOS mobile device management (MDM) remain defaults. They do not automatically become enforced enterprise requirements.

Think of `requirements.toml` as the organization's outer boundary and `config.toml` as the preferred starting position inside that boundary. For example, requirements can permit only cached or disabled web search, while client configuration chooses cached search as the initial setting. A user can choose disabled search, but a supported managed client cannot accept live search when the requirements forbid it.

The same distinction applies to approvals. A requirement can allow `on-request`, while client configuration selects it as the default. That combination still lets a human approve an eligible exception. If exceptions must be impossible, change both files to `never` and forbid the corresponding execution rules.

Some controls also belong outside TOML. Identity-provider policies, single sign-on, endpoint restrictions, firewall rules, approved model access, contractual retention terms, and enterprise data-residency commitments must be enforced through the corresponding administrative, contractual, or operating-system control. Remote or cloud-hosted execution environments also require separately verified controls.

Always keep every supported Codex surface on the latest available release through the organization's approved software-distribution and change-management process. Validate managed settings against the current configuration reference and the schema bundled with the installed release before deployment. Outdated clients may not support or enforce current enterprise requirements consistently.

Where the approved deployment process uses npm, install the current release without pinning a specific Codex release number:

```bash
npm install -g @openai/codex@latest
```

Update managed desktop and IDE installations through the organization's approved application-management process. Do not ask users to bypass endpoint, administrator, or package-registry controls to update their clients.

Permission profiles replace the older `sandbox_mode` and `[sandbox_workspace_write]` configuration. When a deployment uses managed permission profiles, do not mix those older client settings into `config.toml`; define filesystem and command-network boundaries inside the selected permission profile instead.

## Define the regulated operating model

Use this control matrix when translating security and compliance requirements into Codex configuration:

| Governance area | Enterprise requirements | Client defaults and operational controls |
| --- | --- | --- |
| Identity and tenancy | Require enterprise ChatGPT sign-in and, when appropriate, restrict users to verified enterprise workspace IDs. | Apply the correct group policy, configure matching login defaults, and enforce identity-provider controls. |
| Model governance | Enforce an approved model catalog through managed requirements when the organization restricts model selection. | Select approved model and reasoning defaults; use server-side entitlements for actual model authorization. |
| Human oversight | Allow only approved approval policies and human reviewers. | Default to human-reviewed `on-request` execution. |
| Execution boundaries | Allow only approved inspection and workspace permission profiles; exclude full access. | Select the reviewed managed permission profile without mixing legacy sandbox settings. |
| Network and web search | Allow only reviewed web-search modes and restrict command-network access. | Default to cached search and disable command networking. |
| Apps, MCP, and plugins | Deny unapproved standalone and plugin-bundled MCP identities and disable high-risk managed features. | Disable apps by default and enable plugins or connectors only after review. |
| Credentials and secrets | Protect sensitive files through both global managed read denials and the approved permission profile. | Store credentials in the operating-system keyring and restrict inherited shell variables. |
| Data handling | Enforce only supported product requirements; do not invent residency values. | Minimize local history, disable analytics where appropriate, and suppress prompt logging. |
| Monitoring and support | Review managed hooks and change-control rules. | Configure approved OpenTelemetry settings and retain the ability to submit `/feedback`. |
| Production safety | Require approval for sensitive development actions and forbid protected production changes. | Route deployment, infrastructure, and merge actions through approved workflows. |
| Device security | Prefer the elevated Windows sandbox and enforce private-desktop isolation. | Document any approved unelevated exception, compensating controls, and remediation plan. |

The baseline below is supervised, not maximally restrictive. It supports normal engineering work while reserving sensitive actions for human review. Organizations that cannot allow user-approved exceptions should adopt the stricter approval model described later.

## Create the enterprise requirements file

The complete portable managed policy is available as [requirements.toml](regulated_industry_configuration/requirements.toml):

```toml
# Cross-platform regulated-enterprise requirements for the latest Codex release.
# Deploy through enterprise-managed configuration or the system policy path.
# These requirements establish governance boundaries, not user defaults.

allowed_approval_policies = ["on-request", "untrusted"]
allowed_approvals_reviewers = ["user"]
allowed_web_search_modes = ["disabled", "cached"]
default_permissions = "regulated_workspace"
allow_login_shell = false
allow_managed_hooks_only = true
allow_appshots = false
allow_remote_control = false

# Optionally enforce an approved model catalog installed on each managed device.
# model_catalog_json = "/etc/codex/approved-models.json"

# Explicit empty allowlists block standalone and plugin-bundled MCP servers.
# Add only approved, verified server and plugin identities.
mcp_servers = {}
plugins = {}

# Enforce the stronger sandbox and isolated desktop on native Windows clients.
[windows]
allowed_sandbox_implementations = ["elevated"]
sandbox_private_desktop = true

[allowed_permission_profiles]
":read-only" = true
regulated_read_only = true
regulated_workspace = true

[permissions.regulated_read_only]
description = "Inspect approved files without modifying the workspace or using command networking."
extends = ":read-only"

[permissions.regulated_read_only.network]
enabled = false

[permissions.regulated_workspace]
description = "Use the standard workspace sandbox with human oversight, protected secrets, and restricted command networking."
extends = ":workspace"

[permissions.regulated_workspace.filesystem]
glob_scan_max_depth = 6

[permissions.regulated_workspace.filesystem.":workspace_roots"]
".env" = "deny"
".env.local" = "deny"
"**/.env" = "deny"
"**/.env.*" = "deny"
"**/*.env" = "deny"
"**/*.key" = "deny"
"**/*.pem" = "deny"
"**/*.p12" = "deny"
"**/*.pfx" = "deny"

[permissions.regulated_workspace.network]
enabled = false

# Protect sensitive credentials across approved profiles, not only inside roots.
# Native Windows shell reads also require operating-system and endpoint controls.
[permissions.filesystem]
deny_read = [
  "~/.ssh/id_rsa",
  "~/.ssh/id_ed25519",
  "~/.aws/credentials",
  "~/.azure",
  "~/.config/gcloud",
  "~/.kube/config",
  "~/.docker/config.json",
  "~/.npmrc",
  "~/.pypirc",
  "~/.netrc",
  "/**/.env",
  "/**/*.env",
  "/**/*.pem",
  "/**/*.key",
  "/**/*.p12",
  "/**/*.pfx",
]

[features]
browser_use = false
browser_use_external = false
computer_use = false
enable_mcp_apps = false
guardian_approval = false
in_app_browser = false
memories = false
memory_tool = false
plugin_sharing = false
remote_control = false
remote_plugin = false
skill_mcp_dependency_install = false

# Prompt for reviewable engineering actions and forbid high-risk operations.
[rules]
prefix_rules = [
  { pattern = [{ any_of = ["bash", "sh", "zsh", "pwsh", "powershell", "cmd"] }], decision = "prompt", justification = "Nested shell entry points require explicit human review." },
  { pattern = [{ token = "rm" }, { any_of = ["-rf", "-fr", "-Rf", "-fR"] }], decision = "forbidden", justification = "Forced recursive deletion is not allowed." },
  { pattern = [{ token = "Remove-Item" }, { token = "-Recurse" }, { token = "-Force" }], decision = "forbidden", justification = "Forced recursive deletion is not allowed." },
  { pattern = [{ token = "Remove-Item" }, { token = "-Force" }, { token = "-Recurse" }], decision = "forbidden", justification = "Forced recursive deletion is not allowed." },
  { pattern = [{ any_of = ["rm", "rmdir", "del", "Remove-Item"] }], decision = "prompt", justification = "A human must review destructive filesystem operations." },
  { pattern = [{ token = "git" }, { token = "reset" }, { token = "--hard" }], decision = "forbidden", justification = "A hard reset can discard uncommitted work." },
  { pattern = [{ token = "git" }, { any_of = ["commit", "push", "clean", "rebase", "checkout", "switch"] }], decision = "prompt", justification = "A human must approve repository mutations and publication." },
  { pattern = [{ any_of = ["curl", "wget", "Invoke-WebRequest", "Invoke-RestMethod"] }], decision = "prompt", justification = "A human must review attempted data transfer." },
  { pattern = [{ any_of = ["npm", "pnpm", "yarn", "pip", "pip3", "uv"] }, { any_of = ["install", "add"] }], decision = "prompt", justification = "Dependency changes require approved package sources and human review." },
  { pattern = [{ any_of = ["docker", "podman"] }, { any_of = ["run", "pull", "build", "push"] }], decision = "prompt", justification = "Container execution and registry changes require human review." },
  { pattern = [{ any_of = ["ngrok", "localtunnel", "devtunnel"] }], decision = "forbidden", justification = "Public tunneling is not allowed; use an approved internal environment." },
  { pattern = [{ token = "cloudflared" }, { token = "tunnel" }], decision = "forbidden", justification = "Public tunneling is not allowed; use an approved internal environment." },
  { pattern = [{ token = "code" }, { token = "tunnel" }], decision = "forbidden", justification = "Editor tunnels are not allowed without an approved exception." },
  { pattern = [{ token = "ssh" }, { any_of = ["-R", "-L", "-D"] }], decision = "forbidden", justification = "SSH port forwarding and tunnels require an approved external workflow." },
  { pattern = [{ any_of = ["sudo", "su", "runas", "pkexec"] }], decision = "forbidden", justification = "Privilege escalation is not allowed from agent-managed sessions." },
  { pattern = [{ any_of = ["mimikatz", "procdump", "procdump.exe"] }], decision = "forbidden", justification = "Credential extraction and process-memory dumping are not allowed." },
  { pattern = [{ any_of = ["reg", "reg.exe"] }, { any_of = ["add", "delete", "import"] }], decision = "forbidden", justification = "Windows registry changes require approved endpoint-management workflows." },
  { pattern = [{ any_of = ["sc", "sc.exe"] }, { any_of = ["create", "config", "delete"] }], decision = "forbidden", justification = "Windows service changes require approved endpoint-management workflows." },
  { pattern = [{ token = "kubectl" }, { any_of = ["apply", "delete", "patch", "exec", "edit", "port-forward"] }], decision = "forbidden", justification = "Production changes and service exposure require approved deployment pipelines." },
  { pattern = [{ token = "terraform" }, { any_of = ["apply", "destroy"] }], decision = "forbidden", justification = "Infrastructure changes require approved change management." },
  { pattern = [{ token = "gh" }, { token = "pr" }, { token = "merge" }], decision = "forbidden", justification = "Pull-request merges remain human-controlled actions outside Codex." },
]
```

### Explain the organization-wide requirements

The top-level entries define which security-sensitive choices a user or client may make. Keep these entries before the first `[table]` header so TOML does not accidentally place them inside another section.

| Requirement | What it controls | Why the example uses it |
| --- | --- | --- |
| `allowed_approval_policies = ["on-request", "untrusted"]` | Limits the available approval modes to supervised workflows. | Keeps human review available for sensitive actions while still supporting ordinary engineering work. It does not prevent a user from approving every eligible exception. |
| `allowed_approvals_reviewers = ["user"]` | Allows a human user, not an automated reviewer, to decide approval requests. | Makes an identified person responsible for reviewing actions that need an exception. |
| `allowed_web_search_modes = ["disabled", "cached"]` | Allows no search or cached results; live search is unavailable. | Reduces exposure to arbitrary live web content while allowing the organization to choose a stricter no-search posture. |
| `default_permissions = "regulated_workspace"` | Selects the managed starting permission profile. | Gives every managed session a predictable baseline; the profile allowlist separately defines which alternatives are permitted. |
| `allow_login_shell = false` | Rejects login-shell execution. | Avoids loading interactive login-shell behavior when Codex starts a command. |
| `allow_managed_hooks_only = true` | Allows managed hooks while skipping user, project, session, and plugin hooks. | Prevents unreviewed hooks from silently adding commands, integrations, or other behavior. |
| `allow_appshots = false` | Disables Appshots where the client supports that feature. | Avoids unintentionally adding application-screen content to a regulated session. |
| `allow_remote_control = false` | Disables supported device remote control. | Keeps the session on the approved managed device; it does not disable every SSH or remote-access technology. |
| `[windows].allowed_sandbox_implementations = ["elevated"]` | Allows only the stronger native Windows sandbox. | Prevents an unapproved fallback to the weaker `unelevated` implementation. |
| `[windows].sandbox_private_desktop = true` | Requires an isolated desktop for native Windows sandbox processes. | Keeps the stronger user-interface boundary enforced instead of relying only on a client default. |
| `mcp_servers = {}` | Creates an explicit empty allowlist for standalone MCP servers. | Starts with no additional tool integrations until an administrator approves a server's exact identity. |
| `plugins = {}` | Creates an explicit empty allowlist for plugin-bundled MCP servers. | Stops a plugin from reintroducing an unreviewed MCP integration through a different route. |
| `model_catalog_json` when enabled | Points to a protected, locally deployed approved-model catalog. | Restricts the visible Codex model and reasoning choices without claiming to replace backend model authorization. |

An empty MCP or plugin allowlist is different from omitting that requirement. Keep the explicit empty table until the organization approves exact server identities, reachable systems, data access, and change ownership.

### Explain the permission profiles

> **Beta:** Permission profiles are under active development and may change. Verify support against the latest approved client and current configuration reference before using them in a regulated environment.

`[allowed_permission_profiles]` is the complete allowlist. The example permits the built-in `:read-only` profile and two named enterprise profiles; omitted profiles, including unrestricted full access, are unavailable. Setting `default_permissions = "regulated_workspace"` selects one approved profile but does not make the other listed profiles the default.

`[permissions.regulated_read_only]` extends `:read-only` and sets `network.enabled = false`. Use it for investigation or document review when users should not modify the workspace or give spawned commands network access.

`[permissions.regulated_workspace]` extends `:workspace`, so approved development work can continue inside the normal workspace boundary. It adds explicit deny rules for common `.env`, private-key, and certificate files, and it also sets `network.enabled = false`. That setting applies to sandboxed commands; it does not disconnect the Codex client from its authenticated model or management service.

`glob_scan_max_depth = 6` bounds how deeply applicable platforms expand recursive deny patterns before sandbox startup. Larger values can discover more nested files but increase scanning work. Wildcard matches can be platform-dependent or based on files present at startup, so pair them with endpoint and operating-system controls.

`[permissions.filesystem].deny_read` extends protection to common SSH keys, cloud credentials, Kubernetes configuration, container credentials, package-manager settings, and certificate files beyond one workspace. On native Windows, these managed denials apply to direct file tools, but shell subprocess reads require separate operating-system or endpoint restrictions.

### Explain feature restrictions

The `[features]` table pins selected capabilities off in the managed policy. Group the settings by the risk they address:

- `browser_use`, `browser_use_external`, and `in_app_browser` reduce access to interactive browser surfaces and external web content.
- `computer_use`, `remote_control`, and the top-level remote-control requirement limit device interaction outside the approved coding workflow.
- `memories` and `memory_tool` disable persistent memory features that may retain context across sessions.
- `enable_mcp_apps`, `remote_plugin`, `plugin_sharing`, and `skill_mcp_dependency_install` reduce unreviewed connector, plugin, and dependency-installation paths.
- `guardian_approval = false`, together with the human-reviewer allowlist, keeps approval decisions on the supervised user-review path.

Feature support can vary by client and management surface. For example, the `plugin_sharing` restriction applies to supported cloud-managed requirements. Verify the effective feature policy on each deployed client before relying on it.

### Explain command-review rules

`[rules].prefix_rules` examines supported command prefixes before execution. A `prompt` rule asks a human to review an otherwise eligible action; a `forbidden` rule blocks it. The justification explains the security decision to the person reviewing the session.

The rule for `bash`, `sh`, `zsh`, `pwsh`, `powershell`, and `cmd` requires human approval before a nested shell starts. Codex can inspect simple command chains, but scripts using variable expansion, redirection, or control flow may be evaluated as a single shell invocation. `allow_login_shell = false` restricts login-shell behavior; it does not replace approval for nested shell entry points.

| Decision | Examples in the policy | Why it matters |
| --- | --- | --- |
| `prompt` | Nested shell entry points, file deletion, Git changes, data-transfer tools, dependency installation, and container operations. | These actions can be legitimate, but they may publish data, change trusted code, or alter a developer workstation. |
| `forbidden` | Forced recursive deletion, destructive Git reset, public tunnels, privilege escalation, credential extraction, Windows service or registry changes, protected infrastructure changes, and pull-request merges. | These actions exceed the example's approved development boundary and should use a separate authorized workflow. |

Command-prefix rules do not understand every possible shell script, argument variation, or indirect execution path. Treat them as one review layer alongside sandboxing, endpoint controls, identity policy, and repository protections.

## Create the matching client configuration

The complete portable client configuration is available as [config.toml](regulated_industry_configuration/config.toml):

```toml
#:schema https://developers.openai.com/codex/config-schema.json
# Cross-platform device or user defaults for the latest Codex release.
# Pair this file with the corresponding enforced requirements.toml.

approval_policy = "on-request"
approvals_reviewer = "user"
default_permissions = "regulated_workspace"
web_search = "cached"
allow_login_shell = false
model_reasoning_effort = "medium"

# Optional approved-model defaults. Verify model access and install the catalog
# before enabling these settings; enforce the catalog through requirements.
# model = "gpt-5.6-luna"
# model_catalog_json = "/etc/codex/approved-models.json"

forced_login_method = "chatgpt"
cli_auth_credentials_store = "keyring"
mcp_oauth_credentials_store = "keyring"

# Match the managed native Windows sandbox and isolated-desktop requirements.
[windows]
sandbox = "elevated"
sandbox_private_desktop = true

[apps._default]
enabled = false
destructive_enabled = false
open_world_enabled = false

[analytics]
enabled = false

[feedback]
enabled = true

[history]
persistence = "none"

[shell_environment_policy]
inherit = "core"
ignore_default_excludes = false

[shell_environment_policy.filters]
"*PASSWORD*" = "exclude"
"*CREDENTIAL*" = "exclude"
"*PRIVATE*" = "exclude"

[otel]
environment = "regulated-production"
log_user_prompt = false
```

### Explain everyday operating defaults

These top-level settings determine how a supported local client starts. Each value must stay inside any corresponding managed requirement.

| Client setting | Starting behavior | Why the example chooses it |
| --- | --- | --- |
| `approval_policy = "on-request"` | Codex asks before an eligible action needs additional approval. | Supports supervised engineering without requiring a prompt for every ordinary in-sandbox action. |
| `approvals_reviewer = "user"` | Approval requests go to the human user. | Matches the managed reviewer allowlist and makes the review decision visible to a person. |
| `default_permissions = "regulated_workspace"` | Starts in the approved enterprise workspace profile. | Uses the managed filesystem and command-network restrictions without falling back to older sandbox settings. |
| `web_search = "cached"` | Uses indexed cached search results instead of live retrieval. | Preserves limited research capability while remaining inside the managed search allowlist. |
| `allow_login_shell = false` | Starts supported shell tools without login-shell behavior. | Matches the managed shell restriction and avoids an unsupported login-shell request. |
| `model_reasoning_effort = "medium"` | Selects a balanced default reasoning effort. | Provides a predictable starting point for cost, latency, and task complexity; it is not a model-access boundary. |

`on-request` is a supervised-exception model, not an absolute prohibition. When the organization cannot allow a user to approve a sandbox exception, select `never` in both the requirements allowlist and the client configuration.

### Explain identity and credential storage

`forced_login_method = "chatgpt"` selects ChatGPT authentication so the session uses the organization's approved workspace identity and controls rather than an unrelated API-key workflow. Apply this setting through protected managed configuration when users must not change it, and enforce the account boundary through enterprise identity and workspace administration.

`forced_chatgpt_workspace_id` can additionally restrict the active ChatGPT workspace. Add it only after administrators independently verify the organization's real workspace identifier. Never deploy a copied example identifier or assume a user-editable file alone establishes a non-overridable identity boundary.

`cli_auth_credentials_store = "keyring"` requests the operating-system credential store for Codex authentication, and `mcp_oauth_credentials_store = "keyring"` requests the same storage approach for approved MCP OAuth credentials. Review keyring availability and endpoint policy before rollout; these settings do not by themselves approve an MCP integration.

### Explain apps, retention, feedback, and telemetry

`[apps._default]` starts all apps disabled and also disables app actions marked destructive or open-world. These are client defaults, not a substitute for the managed MCP allowlists, feature restrictions, workspace permissions, or an explicit third-party risk review.

`[analytics].enabled = false` disables optional analytics for this machine or profile. `[history].persistence = "none"` avoids saving local session transcripts to `history.jsonl`. Neither setting changes contractual OpenAI data retention, compliance exports, repository audit logs, or other enterprise logging controls.

`[feedback].enabled = true` keeps `/feedback` available so a user can report a problem from the active session and share a feedback ID with support. This helps incident investigation even when local history persistence is disabled. Apply the organization's data-handling policy before including any diagnostic content.

`[otel].environment = "regulated-production"` labels approved OpenTelemetry events with an environment name, while `log_user_prompt = false` avoids exporting raw user prompts through that OpenTelemetry setting. The example does not configure an exporter, collector, token, or external destination; enable those only after the organization approves the complete telemetry pipeline.

### Explain shell environment filtering

`[shell_environment_policy].inherit = "core"` passes a reduced baseline of environment variables to spawned commands. `ignore_default_excludes = false` also activates Codex's built-in exclusions for variable names containing `KEY`, `SECRET`, or `TOKEN`.

`[shell_environment_policy.filters]` adds the supported pattern-based filters for `*PASSWORD*`, `*CREDENTIAL*`, and `*PRIVATE*`. Together, these settings reduce accidental credential exposure to tools and shell commands. Do not combine these filters with the older `exclude` or `include_only` arrays in the same configuration layer.

### See how the two files work together

| Security decision | Managed requirement | Matching client default | Result for the reader |
| --- | --- | --- | --- |
| Human oversight | `allowed_approval_policies` and `allowed_approvals_reviewers` | `approval_policy` and `approvals_reviewer` | The client starts in an approved human-review mode and cannot select a reviewer outside the managed allowlist. |
| Execution boundary | `allowed_permission_profiles` and managed `default_permissions` | `default_permissions` | The client starts in the regulated workspace profile and cannot select an unapproved profile. |
| Native Windows sandbox | `[windows].allowed_sandbox_implementations` and `[windows].sandbox_private_desktop` | `[windows].sandbox` and `[windows].sandbox_private_desktop` | Native Windows sessions use the approved elevated sandbox and isolated desktop without an unapproved compatibility fallback. |
| Web search | `allowed_web_search_modes` | `web_search` | Cached search is selected by default, while live search remains unavailable. |
| Shell behavior | `allow_login_shell = false` | `allow_login_shell = false` | The default behavior and enforced shell restriction remain aligned. |
| Approved models | Optional managed `model_catalog_json` | Optional `model` and `model_reasoning_effort` | Managed catalog restrictions determine available Codex choices; client values select a starting option. |
| Sign-in and credentials | Enterprise identity, protected managed configuration, and endpoint controls | `forced_login_method` and keyring settings | The preferred sign-in and credential stores are explicit, but account enforcement still requires the corresponding administrative controls. |
| Local privacy and observability | Supported managed requirements or separate enterprise controls when available | App defaults, analytics, feedback, history, shell filters, and OpenTelemetry settings | The client starts conservatively, but a user-editable setting becomes non-overridable only when a supported administrative control enforces it. |

Before deployment, verify which source supplies each setting. A client default is convenient and auditable, but it is not automatically an enterprise policy.

## Restrict visible models and reasoning options

Regulated organizations often want approved user groups to see only reviewed models and reasoning levels. Administrators can enforce a local JSON model catalog through managed requirements and assign the policy to the appropriate enterprise users or groups. Keep clients updated so the managed catalog and group-targeted policy remain supported.

These controls have different boundaries:

| Control | Configuration layer | Effect | Limitation |
| --- | --- | --- | --- |
| `model` and `model_reasoning_effort` | Client `config.toml` or managed defaults. | Select the preferred default model and reasoning effort. | Defaults do not independently prevent users from selecting another model. |
| `model_catalog_json` | Managed `requirements.toml`. | Enforce the catalog that controls visible Codex models and reasoning options. | Does not establish server-side model authorization. |
| Workspace and product entitlements | Supported administrative and identity controls. | Govern actual model availability for the relevant account and product surface. | Must be reviewed separately for desktop, CLI, IDE, cloud, and API access. |

Set `model_catalog_json` as a top-level requirement before any TOML table headers. On macOS, a managed requirements file can use:

```toml
# requirements.toml - keep Codex updated through approved device management
model_catalog_json = "/etc/codex/approved-models.json"
```

On native Windows, use the equivalent protected local device path:

```toml
# requirements.toml - keep Codex updated through approved device management
model_catalog_json = 'C:\ProgramData\OpenAI\Codex\approved-models.json'
```

The path must reference a JSON file installed on the local device. An HTTPS URL does not distribute or load the catalog. Cloud-managed requirements can assign the path to an approved user or group, but the organization must still deploy the JSON file through its device-management process and prevent users from modifying it.

Add matching client defaults only after the approved model is available to the enterprise account:

```toml
# config.toml
model = "gpt-5.6-luna"
model_reasoning_effort = "medium"
model_catalog_json = "/etc/codex/approved-models.json"
```

On Windows, replace the catalog path with the single-quoted Windows path shown above. A `model_catalog_json` setting that appears only in client configuration remains a client preference. Put the matching setting in managed requirements when the catalog must be enforced.

### Apply different model policies to enterprise groups

When different teams require different approved models, create a reviewed catalog and managed policy for each risk group. Assign the relevant cloud-managed requirements policy to the intended enterprise users or groups, distribute its matching protected catalog through device management, and verify that pilot users receive the expected model picker. Confirm that users outside the target group retain their intended policy and review source precedence when system, cloud-managed, or MDM requirements also apply.

Group assignment controls which catalog policy a Codex user receives. It does not replace workspace entitlements, backend authorization, or the organization's identity-provider access controls. Managed new-thread model defaults can establish a preferred starting model, but they remain defaults and do not replace an enforced catalog.

### Build an approved model catalog

A Codex model catalog contains complete model definitions, not just model names. Start with the raw catalog exposed by an authorized Codex client, preserve each approved model's full metadata, and remove models or reasoning options that the organization has not reviewed.

From an authorized account, export the current model catalog:

```bash
codex debug models > approved-models.json
```

Review the exported JSON, remove unapproved models and reasoning levels, and confirm each remaining default reasoning level is permitted. Preserve every approved model's required metadata, distribute the reviewed file to the protected path on each target device, and restart Codex after updating it. The catalog is loaded at startup; clients do not automatically reload catalog changes.

The example model is illustrative and must be verified against the organization's approved model list, current entitlement, and deployed client. Maintain the catalog as a reviewed enterprise artifact, refresh it when approved models change, and pilot the policy with the intended user group before wider rollout.

An enforced catalog governs the Codex catalog and model picker. It does not establish a backend authorization boundary or prevent a different product surface or API credential from reaching a model that the backend still authorizes. When a model must be inaccessible rather than hidden from normal selection, require the corresponding server-side administrative or entitlement control and verify its effective behavior.

## Review the major governance decisions

### Identity, workspace, and model access

Bind deployment to the correct enterprise tenant through the organization's identity provider, provisioning process, and approved ChatGPT workspace. Set `forced_login_method = "chatgpt"` through protected managed configuration when the organization must exclude unapproved API-key workflows. When enterprise workspace binding is required, add `forced_chatgpt_workspace_id` only after independently verifying the actual production workspace identifier. Never copy an example, customer, or unverified workspace identifier into production.

Keep `model_reasoning_effort` aligned with latency, cost, and workload needs. For managed catalog restrictions, keep all targeted clients updated and use the supported `model_catalog_json` requirement. Actual model authorization, provider approval, account entitlements, and contractual data-processing conditions still require their own supported administrative controls. Do not invent an unsupported `allowed_models` requirement.

### Permission profiles for different enterprise groups

The reference policy includes an inspection-only `regulated_read_only` profile and a default `regulated_workspace` profile with command networking disabled. Assign the narrowest profile that supports each approved workflow.

Organizations that require access to approved registries or internal services can create a separate reviewed network-enabled profile with explicit domain allowlists. Keep that profile unavailable until its destinations, data flows, ownership, and monitoring are approved. Do not use unrestricted global network allowlists, make an internet-enabled profile the default, or grant write access to protected `.git`, `.agents`, or `.codex` paths without a separately approved security requirement.

### Approval policies and separation of duties

| Operating model | Managed approval policy | Result |
| --- | --- | --- |
| Supervised engineering | `on-request` or `untrusted`, with reviewer `user`. | Sensitive actions can be approved by a human; `prompt` execution rules remain usable. |
| Non-overridable sandbox boundary | `never`, with matching client configuration. | Actions requiring additional approval are rejected instead of presented to a user. |
| Fine-grained control | A supported granular approval configuration. | Some prompt categories can remain available while other categories are rejected; verify support across the deployed client and management surface. |

`on-request` can allow a user to approve execution outside the normal sandbox. It is appropriate only when user-approved exceptions are part of the operating model. When that is unacceptable, set `allowed_approval_policies = ["never"]`, set `approval_policy = "never"`, and replace `prompt` command rules with `forbidden` rules where required.

If you create a stricter no-exception variant, verify that the managed approval policy, client defaults, and command rules consistently reject exceptions.

### Network access, search, and integrations

The example disables network access for sandboxed commands and live web search. It does not disable the Codex client's authenticated model-service connection, enterprise configuration, or account sign-in. Govern those connections through approved identity, proxy, and egress policies.

If engineering teams need package registries, source-control services, or internal APIs, introduce a reviewed network allowlist and test the installed client's managed-network controls. Approve MCP servers and apps individually based on verified ownership, reachable systems, data exposure, and write permissions.

### Data retention, telemetry, and residency

`history.persistence = "none"` reduces local session-history retention. It can also make incident reconstruction harder, so pair the setting with an approved enterprise audit or observability process when your retention policy requires one.

Configure an OpenTelemetry exporter only after the destination, authentication, data classification, retention, and access controls have been approved. The example intentionally omits collector URLs, bearer tokens, workspace identifiers, and other organization-specific values.

Do not assume a TOML setting establishes EU residency or satisfies a contractual residency commitment. Check the current supported residency values and validate applicable data-processing controls separately with the enterprise administrators and legal or security owners.

### Filesystem access as one part of the policy

The example extends the built-in `:workspace` profile for compatibility with ordinary development tools and adds global managed `deny_read` protections for SSH keys, cloud credentials, package-manager configuration, container credentials, and certificate material. A workspace profile can still inherit broader filesystem reads than a dedicated read-isolation policy. When an organization must prevent access to network shares or other directories, define explicit readable roots with a stricter custom profile and enforce the same boundary through operating-system and endpoint controls.

A selected workspace is trusted by its workspace profile. If a prohibited network share is selected as the workspace, the profile includes that share by definition. Prevent prohibited workspace selection through identity, endpoint, operating-system, or file-share policy.

On Windows, managed `deny_read` restrictions protect direct file tools, but shell subprocess reads do not use that same restriction. Wildcard rules are expanded against existing files and use the configured scan depth. The explicit `.env` and `.env.local` entries protect known paths, but deeper files, newly created wildcard-only matches, shell access, and network shares require additional operating-system and endpoint controls.

Command-prefix rules provide an additional review boundary, and the most restrictive matching decision takes precedence. They do not semantically inspect every possible shell script. Do not treat those rules as a replacement for sandbox, endpoint, or identity controls.

## Choose an enterprise-managed deployment approach

| Configuration source | Purpose | What to verify |
| --- | --- | --- |
| Cloud-managed requirements | Assign enforced policy to supported enterprise users or groups. | The intended users receive the correct policy, and other groups retain their expected access. |
| System or device-managed requirements | Install an administrator-controlled `requirements.toml` through approved device management. | Users cannot modify the policy, and the client loads the intended managed source. |
| Managed client defaults | Apply a consistent starting configuration across approved devices. | Each default remains inside the enforced requirements and is not mistaken for a non-overridable control. |
| User configuration | Establish local preferences inside the organization's managed boundaries. | Security-critical restrictions also exist in a supported managed policy or external administrative control. |

Start with a pilot group, verify the effective policy for pilot and non-pilot users, and resolve precedence between supported configuration sources before broader deployment.

### Apply native Windows sandbox settings

Keep the same two shared TOML files on every supported platform. Their `[windows]` tables apply when Codex runs natively on Windows. The managed requirements select the permitted implementation and enforce the isolated desktop:

```toml
[windows]
allowed_sandbox_implementations = ["elevated"]
sandbox_private_desktop = true
```

The matching client configuration selects that approved implementation:

```toml
[windows]
sandbox = "elevated"
sandbox_private_desktop = true
```

`elevated` selects the stronger native Windows sandbox; it does not give the agent unrestricted administrator access. Restricting `allowed_sandbox_implementations` to `elevated` prevents a user from selecting the weaker `unelevated` fallback. Enforcing `sandbox_private_desktop = true` in the requirements keeps user-interface isolation enabled even when a user can change ordinary client settings.

Install system-managed Windows requirements at `%ProgramData%\OpenAI\Codex\requirements.toml` through approved endpoint management, or distribute supported cloud-managed requirements. Verify that users cannot modify the active enterprise policy. Native Windows `deny_read` restrictions protect supported direct file tools, while shell subprocess reads still require operating-system and endpoint controls.

If the stronger sandbox cannot initialize because of an approved endpoint restriction, document the exception, assign a remediation owner, and review compensating controls before allowing the weaker fallback. The exception must change both files:

```toml
# requirements.toml - approved Windows compatibility exception
[windows]
allowed_sandbox_implementations = ["unelevated"]
sandbox_private_desktop = true
```

```toml
# config.toml - matching Windows compatibility exception
[windows]
sandbox = "unelevated"
sandbox_private_desktop = true
```

Allow both implementations only when the organization explicitly approves fallback. Disabling the private desktop is a separate compatibility exception that requires its own security review.

### Apply macOS managed configuration

macOS uses its native Seatbelt sandbox and does not require an invented `[macos]` configuration table. Keep the same shared requirements and client settings, then distribute them through supported administrator-controlled files or mobile device management (MDM):

- `/etc/codex/requirements.toml` supplies system-managed enterprise requirements.
- `/etc/codex/managed_config.toml` supplies managed client defaults.
- `~/.codex/config.toml` contains user-level client preferences inside the approved policy boundary.

For MDM, use the `com.openai.codex` preference domain with these base64-encoded TOML payloads:

- `requirements_toml_base64` provides enterprise requirements.
- `config_toml_base64` provides managed client defaults.

MDM-managed defaults have precedence over system-managed defaults and user configuration. Requirements follow their separately documented managed-requirements precedence. A managed default does not become a non-overridable security boundary unless the corresponding supported requirement or external enterprise control also enforces it.

### Compare platform-specific deployment

| Deployment concern | Native Windows | macOS |
| --- | --- | --- |
| Sandbox implementation | Require the stronger `elevated` implementation and an isolated desktop. | Use the native Seatbelt sandbox; no separate `[macos]` table is required. |
| Enforced system policy | `%ProgramData%\OpenAI\Codex\requirements.toml` or supported cloud management. | `/etc/codex/requirements.toml`, supported cloud management, or macOS MDM. |
| Managed client defaults | Use the supported managed configuration source for the Windows deployment. | Use `/etc/codex/managed_config.toml` or the MDM `config_toml_base64` payload. |
| Approved exceptions | Permit `unelevated` only after documented security review and compensating controls. | Resolve compatibility issues through supported device management and endpoint controls. |

Use the latest approved Codex release on both platforms and validate the actual managed source, effective settings, and endpoint restrictions before expanding the rollout.

## Validate the deployment and rollout

Confirm that the two reference files agree on approval policies, permission profiles, human review, and command-network access. Reviewing example files does not prove that a managed device enforces the intended policy.

Validate the policy that actually reaches each managed device. Export or inspect the active requirements in the supported management interface, compare them with the intended files, and identify the effective cloud-managed, system, or MDM source. Do not treat an email attachment, copied chat snippet, cached policy fragment, or reconstructed configuration as proof of the active enterprise policy.

After deploying a pilot:

1. Confirm every local client uses the latest approved release and can enforce the deployed policy.
2. Open `/debug-config` and verify the active requirements source, approval settings, selected permission profile, and applicable platform-specific settings.
3. Confirm the correct enterprise authentication flow, workspace, credential store, and permitted integrations.
4. If a managed model catalog is enabled, verify the active policy source, approved models, reasoning levels, protected catalog path, and backend model entitlements.
5. Check web-search mode, command networking, analytics, local history, and approved telemetry settings.
6. Validate that sensitive development actions prompt for review and protected production actions are forbidden.
7. On native Windows, confirm the elevated sandbox and private desktop initialize, or verify that an approved fallback has compensating controls and a remediation owner.
8. On macOS, verify the active system or MDM requirements source, managed-default precedence, and supported Seatbelt sandbox enforcement.
9. If investigation is required, run `/feedback` in the active session and share the resulting feedback ID through the approved support process.

Do not include raw credentials, customer data, confidential source code, or other sensitive material in support messages.

## Adapt the baseline to the organization

Use one managed baseline as the starting point, then create reviewed variations for distinct risk groups. A software engineering group might permit cached search and approved repository integrations, while a production-support group could use read-only permissions and a no-exception approval policy.

Review each change across the full operating model: identity, human oversight, execution, network access, integrations, sensitive data, auditability, and endpoint controls. Keep all Codex surfaces on the latest approved release, synchronize the two shared files, avoid unverified identifiers, inspect the effective managed policy, and validate any Windows-only settings added during deployment.

## References

- [Managed configuration and enterprise requirements](https://learn.chatgpt.com/docs/enterprise/managed-configuration)
- [Enterprise login and workspace authentication controls](https://learn.chatgpt.com/docs/auth#enforce-a-login-method-or-workspace)
- [Workspace model availability and administrative controls](https://learn.chatgpt.com/docs/enterprise/workspace-model-availability)
- [Permission profiles and execution boundaries](https://learn.chatgpt.com/docs/permissions)
- [Agent approvals, sandboxing, and security](https://learn.chatgpt.com/docs/agent-approvals-security)
- [Native Windows sandbox](https://learn.chatgpt.com/docs/windows/windows-sandbox)
- [Configuration reference](https://learn.chatgpt.com/docs/config-file/config-reference)
- [Advanced configuration and shell environment policy](https://learn.chatgpt.com/docs/config-file/config-advanced)
- [Managed execution rules](https://learn.chatgpt.com/docs/agent-configuration/rules)